In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score


In [3]:
MODEL_READY_CSV = Path("../data/model_ready/match_by_match.csv")

In [4]:
dataset = pd.read_csv(MODEL_READY_CSV)

In [5]:
dataset.shape

(1243, 31)

In [6]:
dataset.head()

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,...,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue
0,335982,2008-04-18,"M Chinnaswamy Stadium, Bengaluru",Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Kolkata Knight Riders,won by 140 runs,222,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,335983,2008-04-19,"Punjab Cricket Association IS Bindra Stadium, ...",Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,Chennai Super Kings,won by 33 runs,240,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,335984,2008-04-19,"Arun Jaitley Stadium, Delhi",Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,Delhi Capitals,won by 9 wickets,129,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,335986,2008-04-20,"Eden Gardens, Kolkata",Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,bat,Kolkata Knight Riders,won by 5 wickets,110,...,222.0,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,335985,2008-04-20,"Wankhede Stadium, Mumbai",Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,won by 5 wickets,165,...,82.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
dataset.tail(5)

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,...,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue
1238,1529313,2026-05-24,"Eden Gardens, Kolkata",Delhi Capitals,Kolkata Knight Riders,Kolkata Knight Riders,field,Delhi Capitals,won by 40 runs,203,...,180.6,173.2,0.490,0.409,0.565,0.500,160.73,0.563,0.200,0.564
1239,1535462,2026-05-26,"Himachal Pradesh Cricket Association Stadium, ...",Royal Challengers Bengaluru,Gujarat Titans,Gujarat Titans,field,Royal Challengers Bengaluru,won by 92 runs,254,...,202.2,157.6,0.550,0.486,0.690,0.500,172.81,0.438,0.667,0.000
1240,1535463,2026-05-27,Maharaja Yadavindra Singh International Cricke...,Rajasthan Royals,Sunrisers Hyderabad,Sunrisers Hyderabad,field,Rajasthan Royals,won by 47 runs,243,...,184.4,183.8,0.545,0.383,0.492,0.359,210.90,0.750,1.000,0.000
1241,1535464,2026-05-29,Maharaja Yadavindra Singh International Cricke...,Rajasthan Royals,Gujarat Titans,Rajasthan Royals,bat,Gujarat Titans,won by 7 wickets,214,...,201.2,175.8,0.545,0.383,0.667,0.500,212.33,0.600,1.000,0.000
1242,1535465,2026-05-31,"Narendra Modi Stadium, Ahmedabad",Gujarat Titans,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,won by 5 wickets,155,...,207.4,194.8,0.667,0.500,0.550,0.486,176.40,0.469,0.600,0.429


In [8]:
dataset["winner"].value_counts()

winner
Mumbai Indians                 155
Chennai Super Kings            148
Royal Challengers Bengaluru    143
Kolkata Knight Riders          140
Punjab Kings                   126
Delhi Capitals                 125
Rajasthan Royals               123
Sunrisers Hyderabad            102
Gujarat Titans                  47
Lucknow Super Giants            34
Deccan Chargers                 29
Rising Pune Supergiants         15
Gujarat Lions                   13
Pune Warriors                   12
Kochi Tuskers Kerala             6
Name: count, dtype: int64

In [9]:
dataset.isna().sum()

match_id                                0
date                                    0
venue                                   0
team1                                   0
team2                                   6
toss_winner                             0
toss_decision                           0
winner                                 25
result                                 25
team1_score                             0
team2_score                             0
team1_players                           0
team2_players                           0
team1_total_wins_against_team2          0
team2_total_wins_against_team1          0
team1_wins_against_team2_last_three     0
team2_wins_against_team1_last_three     0
team1_form_last_5                       0
team2_form_last_5                       0
team1_last_5_avg_score                  0
team1_last_5_runs_conceded              0
team2_last_5_avg_score                  0
team2_last_5_runs_conceded              0
team1_chasing_win_rate            

In [10]:
dataset = dataset.dropna(subset=['winner'])

dataset.shape

(1218, 31)

In [11]:
def get_winner_slot(row):
    if row['winner'] == row['team1']:
        return 1
    else:
        return 0

In [12]:
def get_toss_winner_slot(row):
    if row['toss_winner'] == row['team1']:
        return 1
    else:
        return 0

In [13]:
dataset['winner_slot'] = dataset.apply(get_winner_slot, axis=1)
dataset['toss_winner_slot'] = dataset.apply(get_toss_winner_slot, axis=1)

dataset.head(5)

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,...,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,winner_slot,toss_winner_slot
0,335982,2008-04-18,"M Chinnaswamy Stadium, Bengaluru",Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Kolkata Knight Riders,won by 140 runs,222,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1,335983,2008-04-19,"Punjab Cricket Association IS Bindra Stadium, ...",Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,Chennai Super Kings,won by 33 runs,240,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
2,335984,2008-04-19,"Arun Jaitley Stadium, Delhi",Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,Delhi Capitals,won by 9 wickets,129,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,335986,2008-04-20,"Eden Gardens, Kolkata",Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,bat,Kolkata Knight Riders,won by 5 wickets,110,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
4,335985,2008-04-20,"Wankhede Stadium, Mumbai",Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,won by 5 wickets,165,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


Saving copy of the dataset before the dropping of the essential columns which will be used during model training

In [14]:
dataset_pre_drop = dataset.copy()

### Using `no_score` dataset
Removing categorical features and teams' score before the model training

In [15]:
dataset = dataset.drop(["date", "venue", "result", "toss_winner",
                        "toss_decision", "team1", "team2",
                        "team1_score", 
                        "team2_score", 
                        "team1_players", 
                        "team2_players", "winner"], axis=1)

dataset = dataset.set_index("match_id")

dataset.head(5)

,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,winner_slot,toss_winner_slot
match_id,,,,,,,,,,,,,,,,,,,,
335982,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
335983,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
335984,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335986,0,0,0,0,0,1,0.0,0.0,222.0,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335985,0,0,0,0,0,0,0.0,0.0,82.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


We have removed the both the teams' scores because we didn't have information about their score before the match ever started.

Moving the `winner_slot` column to the front while keeping all other columns in their original order.

In [16]:
cols = list(dataset.columns)
cols.remove("winner_slot")
cols = ["winner_slot"] + cols
dataset = dataset[cols]
dataset.columns

Index(['winner_slot', 'team1_total_wins_against_team2',
       'team2_total_wins_against_team1', 'team1_wins_against_team2_last_three',
       'team2_wins_against_team1_last_three', 'team1_form_last_5',
       'team2_form_last_5', 'team1_last_5_avg_score',
       'team1_last_5_runs_conceded', 'team2_last_5_avg_score',
       'team2_last_5_runs_conceded', 'team1_chasing_win_rate',
       'team1_defending_win_rate', 'team2_chasing_win_rate',
       'team2_defending_win_rate', 'venue_avg_score', 'chasing_win_rate_venue',
       'team1_win_rate_at_venue', 'team2_win_rate_at_venue',
       'toss_winner_slot'],
      dtype='str')

In [17]:
dataset.head(5)

,winner_slot,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,toss_winner_slot
match_id,,,,,,,,,,,,,,,,,,,,
335982,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
335983,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
335984,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
335986,0,0,0,0,0,0,1,0.0,0.0,222.0,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
335985,0,0,0,0,0,0,0,0.0,0.0,82.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


Setting the first column (`winner_slot`) as target and other features as input columns for the model training

In [18]:
target_value = [0]
print(dataset.columns[target_value])

Index(['winner_slot'], dtype='str')


In [19]:
input_values = range(1, dataset.columns.shape[0])

print(dataset.columns[input_values])

Index(['team1_total_wins_against_team2', 'team2_total_wins_against_team1',
       'team1_wins_against_team2_last_three',
       'team2_wins_against_team1_last_three', 'team1_form_last_5',
       'team2_form_last_5', 'team1_last_5_avg_score',
       'team1_last_5_runs_conceded', 'team2_last_5_avg_score',
       'team2_last_5_runs_conceded', 'team1_chasing_win_rate',
       'team1_defending_win_rate', 'team2_chasing_win_rate',
       'team2_defending_win_rate', 'venue_avg_score', 'chasing_win_rate_venue',
       'team1_win_rate_at_venue', 'team2_win_rate_at_venue',
       'toss_winner_slot'],
      dtype='str')


### Splitting the dataset into train and test

In [20]:
x = dataset.iloc[:, input_values]
y = dataset.iloc[:, target_value]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=999)

### Saving the dataset split which would be used in the model interpretation

In [21]:
joblib.dump(X_train, "../data/splits/no_score/X_train.pkl")
joblib.dump(X_test, "../data/splits/no_score/X_test.pkl")
joblib.dump(y_train, "../data/splits/no_score/y_train.pkl")
joblib.dump(y_test, "../data/splits/no_score/y_test.pkl")

['../data/splits/no_score/y_test.pkl']

### Defining a function for training the model with GridSearch Hypertuning 

In [22]:
def train_model(pipeline, param_grid, model_path):
    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print(grid.best_params_)
    print(grid.best_score_)

    joblib.dump(grid.best_estimator_, model_path)

    return grid.best_estimator_

### Defining a function for evaluating the classifiers

In [23]:
def evaluate_classifier(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"Precision-Recall AUC: {average_precision_score(y_test, y_prob):.4f}")

    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

### Training and saving machine learning models.

#### Decision Tree Classifier

In [24]:
pipeline = Pipeline([
    ("classifier", DecisionTreeClassifier(random_state=999))
])

param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 5, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": [None, "sqrt", "log2"]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/no_score/decision_tree.pkl"
)

evaluate_classifier(best_model, X_test, y_test)


{'classifier__criterion': 'gini', 'classifier__max_depth': 10, 'classifier__max_features': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5}
0.5317895849854614

Accuracy: 0.4549
ROC-AUC: 0.4954
Precision-Recall AUC: 0.4064

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.25      0.35       144
           1       0.41      0.75      0.53       100

    accuracy                           0.45       244
   macro avg       0.50      0.50      0.44       244
weighted avg       0.52      0.45      0.42       244

Confusion Matrix:
[[ 36 108]
 [ 25  75]]


#### Random Forest Classifier

In [25]:
pipeline = Pipeline([
    ("classifier", RandomForestClassifier(random_state=999))
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/no_score/random_forest.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

c:\venvs\datasci\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'classifier__criterion': 'entropy', 'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
0.5092149088025376

Accuracy: 0.5779
ROC-AUC: 0.5920
Precision-Recall AUC: 0.4953

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.71      0.66       144
           1       0.48      0.39      0.43       100

    accuracy                           0.58       244
   macro avg       0.55      0.55      0.55       244
weighted avg       0.57      0.58      0.57       244

Confusion Matrix:
[[102  42]
 [ 61  39]]


#### Logistic Regression

In [26]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs", "liblinear"],
    "classifier__max_iter": [1000],
    "classifier__class_weight": [None, "balanced"]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/no_score/logistic_regression.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

{'classifier__C': 0.01, 'classifier__class_weight': None, 'classifier__max_iter': 1000, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
0.51949246629659

Accuracy: 0.5164
ROC-AUC: 0.4821
Precision-Recall AUC: 0.4107

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.76      0.65       144
           1       0.33      0.17      0.22       100

    accuracy                           0.52       244
   macro avg       0.45      0.46      0.44       244
weighted avg       0.47      0.52      0.47       244

Confusion Matrix:
[[109  35]
 [ 83  17]]


c:\venvs\datasci\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


#### Linear Regression (With Threshold)

In [27]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

param_grid = {
    "regressor__fit_intercept": [True, False],
    "regressor__positive": [True, False]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/no_score/linear_regression.pkl"
)

y_score = best_model.predict(X_test)
y_pred = (y_score >= 0.5).astype(int)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_score):.4f}")
print(f"Precision-Recall AUC: {average_precision_score(y_test, y_score):.4f}")

print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

{'regressor__fit_intercept': True, 'regressor__positive': True}
nan

Accuracy: 0.5492
ROC-AUC: 0.5288
Precision-Recall AUC: 0.4268

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.74      0.66       144
           1       0.42      0.27      0.33       100

    accuracy                           0.55       244
   macro avg       0.51      0.51      0.49       244
weighted avg       0.52      0.55      0.52       244

Confusion Matrix:
[[107  37]
 [ 73  27]]


c:\venvs\datasci\Lib\site-packages\sklearn\model_selection\_search.py:1234: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan]
  warnings.warn(


#### Support Vector Classifier

In [28]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(probability=True, random_state=0))
])

param_grid = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto", 0.1, 0.01]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/no_score/support_vector_classifier.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

{'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
0.5359344435633095

Accuracy: 0.5902
ROC-AUC: 0.4996
Precision-Recall AUC: 0.4000

Classification Report:
              precision    recall  f1-score   support

           0       0.59      1.00      0.74       144
           1       0.00      0.00      0.00       100

    accuracy                           0.59       244
   macro avg       0.30      0.50      0.37       244
weighted avg       0.35      0.59      0.44       244

Confusion Matrix:
[[144   0]
 [100   0]]


c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\venvs\datasci\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\venvs\datasci\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\venvs\datasci\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wit

#### XGBoost Classifier (i.e. eXtreme Gradient Boosting Classifier) 

In [29]:
pipeline = Pipeline([
    ("classifier", XGBClassifier())
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__max_depth": [3, 5, 7],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
    "classifier__min_child_weight": [1, 3, 5]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/no_score/xg_boost.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

{'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.2, 'classifier__max_depth': 5, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
0.5164208300290773

Accuracy: 0.5779
ROC-AUC: 0.5859
Precision-Recall AUC: 0.5007

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.63      0.64       144
           1       0.49      0.50      0.49       100

    accuracy                           0.58       244
   macro avg       0.57      0.57      0.57       244
weighted avg       0.58      0.58      0.58       244

Confusion Matrix:
[[91 53]
 [50 50]]


### Choosing the best model for `no_score` dataset
The best model choosen is `RandomForestClassifer` because 
- It has highest classification accuracy
- It has highest macro f1 score
- It has balanced class predictions (unlike SVC which have higher ROC-AUC and Precision-Recall AUC)

The model is saved in the `project_root/models/production/no_score`

### Using `with_score` dataset
We have not dropped the team1 score in this case because we are training the model by using first innings' score as the feature

In [30]:
dataset = dataset_pre_drop.copy()

dataset = dataset.drop(["date", "venue", "result", "toss_winner",
                        "toss_decision", "team1", "team2",
                        "team2_score", 
                        "team1_players", 
                        "team2_players", "winner"], axis=1)

dataset = dataset.set_index("match_id")

dataset.head(5)

,team1_score,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,...,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,winner_slot,toss_winner_slot
match_id,,,,,,,,,,,,,,,,,,,,,
335982,222,0,0,0,0,0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
335983,240,0,0,0,0,0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
335984,129,0,0,0,0,0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335986,110,0,0,0,0,0,1,0.0,0.0,222.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335985,165,0,0,0,0,0,0,0.0,0.0,82.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


In [31]:
cols = list(dataset.columns)
cols.remove("winner_slot")
cols = ["winner_slot"] + cols
dataset = dataset[cols]
dataset.columns

Index(['winner_slot', 'team1_score', 'team1_total_wins_against_team2',
       'team2_total_wins_against_team1', 'team1_wins_against_team2_last_three',
       'team2_wins_against_team1_last_three', 'team1_form_last_5',
       'team2_form_last_5', 'team1_last_5_avg_score',
       'team1_last_5_runs_conceded', 'team2_last_5_avg_score',
       'team2_last_5_runs_conceded', 'team1_chasing_win_rate',
       'team1_defending_win_rate', 'team2_chasing_win_rate',
       'team2_defending_win_rate', 'venue_avg_score', 'chasing_win_rate_venue',
       'team1_win_rate_at_venue', 'team2_win_rate_at_venue',
       'toss_winner_slot'],
      dtype='str')

### Splitting the dataset into train and test. But this time we have added new feature `team1_score`.

In [32]:
target_value = [0]
input_values = range(1, dataset.columns.shape[0])

x = dataset.iloc[:, input_values]
y = dataset.iloc[:, target_value]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=999)

### Saving the dataset split which would be used in the model interpretation

In [33]:
joblib.dump(X_train, "../data/splits/with_score/X_train.pkl")
joblib.dump(X_test, "../data/splits/with_score/X_test.pkl")
joblib.dump(y_train, "../data/splits/with_score/y_train.pkl")
joblib.dump(y_test, "../data/splits/with_score/y_test.pkl")

['../data/splits/with_score/y_test.pkl']

### Training and saving machine learning models.

#### Decision Tree Classifier

In [34]:
pipeline = Pipeline([
    ("classifier", DecisionTreeClassifier(random_state=999))
])

param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 5, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": [None, "sqrt", "log2"]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/with_score/decision_tree.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

{'classifier__criterion': 'entropy', 'classifier__max_depth': 5, 'classifier__max_features': None, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 2}
0.6540311921755221

Accuracy: 0.6066
ROC-AUC: 0.6554
Precision-Recall AUC: 0.5333

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.56      0.63       144
           1       0.52      0.67      0.58       100

    accuracy                           0.61       244
   macro avg       0.61      0.62      0.61       244
weighted avg       0.63      0.61      0.61       244

Confusion Matrix:
[[81 63]
 [33 67]]


#### Random Forest Classifier

In [35]:
pipeline = Pipeline([
    ("classifier", RandomForestClassifier(random_state=999))
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"],
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/with_score/random_forest.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

c:\venvs\datasci\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'classifier__criterion': 'entropy', 'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 100}
0.6837324874438276

Accuracy: 0.6475
ROC-AUC: 0.7049
Precision-Recall AUC: 0.6393

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.64      0.68       144
           1       0.56      0.66      0.61       100

    accuracy                           0.65       244
   macro avg       0.64      0.65      0.64       244
weighted avg       0.66      0.65      0.65       244

Confusion Matrix:
[[92 52]
 [34 66]]


#### Logisitic Regression

In [36]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs", "liblinear"],
    "classifier__max_iter": [1000],
    "classifier__class_weight": [None, "balanced"]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/with_score/logistic_regression.pkl"
)

evaluate_classifier(best_model, X_test, y_test)

{'classifier__C': 0.01, 'classifier__class_weight': None, 'classifier__max_iter': 1000, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
0.6765159925984668

Accuracy: 0.6680
ROC-AUC: 0.7214
Precision-Recall AUC: 0.6585

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.74      0.72       144
           1       0.60      0.57      0.58       100

    accuracy                           0.67       244
   macro avg       0.66      0.65      0.65       244
weighted avg       0.67      0.67      0.67       244

Confusion Matrix:
[[106  38]
 [ 43  57]]


c:\venvs\datasci\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


#### Linear Regression (With Threshold)

In [37]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

param_grid = {
    "regressor__fit_intercept": [True, False],
    "regressor__positive": [True, False]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/with_score/linear_regression.pkl"
)

y_score = best_model.predict(X_test)
y_pred = (y_score >= 0.5).astype(int)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_score):.4f}")
print(f"Precision-Recall AUC: {average_precision_score(y_test, y_score)}")

print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

{'regressor__fit_intercept': True, 'regressor__positive': True}
nan

Accuracy: 0.6762
ROC-AUC: 0.7276
Precision-Recall AUC: 0.6744445192008284

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.74      0.73       144
           1       0.61      0.59      0.60       100

    accuracy                           0.68       244
   macro avg       0.66      0.66      0.66       244
weighted avg       0.67      0.68      0.68       244

Confusion Matrix:
[[106  38]
 [ 41  59]]


c:\venvs\datasci\Lib\site-packages\sklearn\model_selection\_search.py:1234: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan]
  warnings.warn(


#### Support Vector Classifier

In [38]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(probability=True, random_state=999))
])

param_grid = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto", 0.1, 0.01]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/with_score/support_vector_classifier.pkl"
)

evaluate_classifier(best_model, X_test, y_test)


{'classifier__C': 1, 'classifier__gamma': 0.01, 'classifier__kernel': 'rbf'}
0.6724134284959027

Accuracy: 0.6557
ROC-AUC: 0.7228
Precision-Recall AUC: 0.6575

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.70      0.71       144
           1       0.58      0.59      0.58       100

    accuracy                           0.66       244
   macro avg       0.64      0.65      0.65       244
weighted avg       0.66      0.66      0.66       244

Confusion Matrix:
[[101  43]
 [ 41  59]]


c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\venvs\datasci\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


#### XGBoost Classifier (i.e. eXtreme Gradient Boosting Classifier) 

In [39]:
pipeline = Pipeline([
    ("classifier", XGBClassifier())
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__max_depth": [3, 5, 7],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
    "classifier__min_child_weight": [1, 3, 5]
}

best_model = train_model(
    pipeline=pipeline,
    param_grid=param_grid,
    model_path="../models/experiments/with_score/xg_boost.pkl"
)

evaluate_classifier(best_model, X_test, y_test)


{'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 3, 'classifier__min_child_weight': 3, 'classifier__n_estimators': 100, 'classifier__subsample': 1.0}
0.6868411313772138

Accuracy: 0.6393
ROC-AUC: 0.7176
Precision-Recall AUC: 0.6704

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.62      0.67       144
           1       0.55      0.67      0.60       100

    accuracy                           0.64       244
   macro avg       0.64      0.64      0.64       244
weighted avg       0.66      0.64      0.64       244

Confusion Matrix:
[[89 55]
 [33 67]]


### Choosing the best model for `with_score` dataset
The best model choosen is `RandomForestClassifier` because 
- It has highest classification accuracy
- It has highest macro f1 score
- It has balanced confusion matrix
- Precision and Recall are balanced

#### Why SVC, Logistic Regression and Linear Regression were not choosen despite having best 'ROC-AUC' and 'Precision-Recall AUC'?
- **SVC:** Support Vector Classifier ranks observations well but produces a less effective final classifier as its default decision threshold.
- **Logistic Regression:** Logistic Regression also offers better overall during the threshold tuning. But Random Forest offers better f1 score and accuracy at class predictions which matters more in deployment settings.
- **Linear Regression:** Linear Regression had a good performance, but essentially it works as regression model, rather than classifier, making it less calibrated for binary classification.

The model is saved in the `project_root/models/production/with_score`